# Query Fabric Graph with GQL over REST

This developer-focused notebook uses the Contoso product review graph created by `infra/create-fabric-graph.py`.

Fabric exposes one stateless query API: send GQL in a JSON request to `executeQuery`. The same GQL can also be pasted into the Fabric Graph code editor for interactive development.

## Step 1: Load configuration and authentication

Load resource identifiers written to `.env` during provisioning. `InteractiveBrowserCredential` signs you in to the Fabric tenant through the browser and obtains a delegated Fabric token without storing or displaying it in the notebook.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **ignite26-prel42** environment with the path **.venv\Scripts\python.exe**.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os

import httpx
from azure.identity import InteractiveBrowserCredential
from dotenv import load_dotenv
from rich.console import Console

load_dotenv(override=True)

console = Console()
fabric_tenant_id = os.environ["FABRIC_TENANT_ID"]
fabric_workspace_id = os.environ["FABRIC_WORKSPACE_ID"]
fabric_graph_id = os.environ["FABRIC_GRAPH_ID"]
fabric_scope = "https://api.fabric.microsoft.com/.default"
gql_endpoint = (
    "https://api.fabric.microsoft.com/v1/workspaces/"
    f"{fabric_workspace_id}/GraphModels/{fabric_graph_id}/executeQuery?preview=true"
)
credential = InteractiveBrowserCredential(tenant_id=fabric_tenant_id)

console.print(f"Graph endpoint configured for workspace {fabric_workspace_id}")

## Step 2: Execute GQL through REST

Send a GQL query to Fabric as `POST {"query": "..."}` and display the JSON response.

In [ ]:
# This query finds negative feature mentions connected to product reviews.
query = """
MATCH (p:`Product`)-[:`HAS_REVIEW`]->(r:`Review`)-[m:`MENTIONS`]->(f:`Feature`)
FILTER m.sentiment = 'negative'
RETURN p.name AS productName,
       r.rating AS rating,
       f.featureName AS feature,
       m.sentiment AS sentiment,
       m.confidence AS confidence
ORDER BY confidence DESC
LIMIT 20
""".strip()


token = credential.get_token(fabric_scope).token
response = httpx.post(
    gql_endpoint,
    headers={"Authorization": f"Bearer {token}"},
    json={"query": query},
    timeout=120.0,
)
response.raise_for_status()

console.print_json(data=response.json())

## ✅ Mission complete

**What you built:**

* ✓ **GQL query over REST**: A query against the product review graph through the Fabric `executeQuery` API.
* ✓ **Feature-level sentiment**: Negative feature mentions from product reviews, with the confidence of each mention.

➡️ Return to lab instructions.